In [1]:
# Test Database Connection
import psycopg2
from sqlalchemy import create_engine
import pandas as pd

# Connection string
DB_URL = "postgresql://localhost:5432/pollutrace_db"

# Test connection
try:
    engine = create_engine(DB_URL)
    conn = engine.connect()
    print("Connected to PostgreSQL successfully")
    conn.close()
except Exception as e:
    print("Connection failed:", e)

Connected to PostgreSQL successfully


In [ ]:
# import city pollution data 
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="pollutrace_db"
)
cur = conn.cursor()

# Drop existing tables and recreate with lowercase columns
cur.execute("DROP TABLE IF EXISTS city_pollution CASCADE;")
cur.execute("DROP TABLE IF EXISTS lstm_predictions CASCADE;")
cur.execute("DROP TABLE IF EXISTS source_analysis CASCADE;")
cur.execute("DROP TABLE IF EXISTS anomalies CASCADE;")
cur.execute("DROP TABLE IF EXISTS model_metrics CASCADE;")

# Table 1 - all lowercase column names
cur.execute("""
    CREATE TABLE city_pollution (
        id          SERIAL PRIMARY KEY,
        city        VARCHAR(50),
        date        DATE,
        pm25        FLOAT,
        pm10        FLOAT,
        no          FLOAT,
        no2         FLOAT,
        nox         FLOAT,
        nh3         FLOAT,
        so2         FLOAT,
        co          FLOAT,
        ozone       FLOAT,
        benzene     FLOAT,
        month       INT,
        season      VARCHAR(20),
        day_of_week INT,
        year        INT
    );
""")

cur.execute("""
    CREATE TABLE lstm_predictions (
        id              SERIAL PRIMARY KEY,
        city            VARCHAR(50),
        date            DATE,
        predicted_pm25  FLOAT,
        actual_pm25     FLOAT,
        rmse            FLOAT,
        mae             FLOAT,
        r2_score        FLOAT,
        created_at      TIMESTAMP DEFAULT NOW()
    );
""")

cur.execute("""
    CREATE TABLE source_analysis (
        id              SERIAL PRIMARY KEY,
        city            VARCHAR(50),
        top_source_1    VARCHAR(50),
        top_source_2    VARCHAR(50),
        top_source_3    VARCHAR(50),
        pm10_importance FLOAT,
        co_importance   FLOAT,
        nh3_importance  FLOAT,
        confidence      FLOAT,
        created_at      TIMESTAMP DEFAULT NOW()
    );
""")

cur.execute("""
    CREATE TABLE anomalies (
        id                   SERIAL PRIMARY KEY,
        city                 VARCHAR(50),
        date                 DATE,
        pm25                 FLOAT,
        reconstruction_error FLOAT,
        threshold            FLOAT,
        is_anomaly           BOOLEAN,
        severity             VARCHAR(20),
        created_at           TIMESTAMP DEFAULT NOW()
    );
""")

cur.execute("""
    CREATE TABLE model_metrics (
        id          SERIAL PRIMARY KEY,
        city        VARCHAR(50),
        model_type  VARCHAR(50),
        rmse        FLOAT,
        mae         FLOAT,
        r2_score    FLOAT,
        confidence  FLOAT,
        trained_at  TIMESTAMP DEFAULT NOW()
    );
""")

conn.commit()
cur.close()
conn.close()
print("All 5 tables recreated with lowercase columns successfully")

All 5 tables recreated with lowercase columns successfully


In [ ]:
# import model metrics
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql://localhost:5432/pollutrace_db")

df = pd.read_csv('../data/processed/city_daily_clean.csv')
df['date'] = pd.to_datetime(df['date'])

cols = ['city', 'date', 'PM25', 'PM10', 'NO', 'NO2', 'NOx',
        'NH3', 'SO2', 'CO', 'Ozone', 'Benzene',
        'month', 'season', 'day_of_week', 'year']

df = df[cols]

# Rename all columns to lowercase to match database
df.columns = [c.lower() for c in df.columns]

df.to_sql('city_pollution', engine, if_exists='append', index=False)
print("city_pollution imported:", len(df), "rows")
print(df.head())

city_pollution imported: 19018 rows
        city       date       pm25       pm10         no        no2  \
0  Bengaluru 2010-01-01  45.680481  89.866905   4.014167  19.772083   
1  Bengaluru 2010-01-02  45.680481  89.866905   5.620625  22.989583   
2  Bengaluru 2010-01-03  51.230321  92.751640   6.998417  30.434917   
3  Bengaluru 2010-01-04  51.230321  92.751640  11.967222  31.255972   
4  Bengaluru 2010-01-05  51.230321  92.751640  12.011667  33.944872   

         nox        nh3       so2        co      ozone   benzene  month  \
0  23.767917  26.069167  1.652000  0.699583  36.084583  2.650500      1   
1  28.594375  25.741042  2.825000  0.695625  35.196042  2.636278      1   
2  37.413889  15.048651  1.764325  0.771427  40.949833  3.941667      1   
3  43.212083  24.080595  6.145159  0.610364  35.175556  4.144000      1   
4  45.929145  23.825179  4.333275  0.626358  46.480353  3.191424      1   

   season  day_of_week  year  
0  Winter            4  2010  
1  Winter            5  

In [7]:
# Import source analysis
metrics_data = {
    'city': ['Delhi', 'Bengaluru', 'Mumbai', 'Hyderabad'],
    'model_type': ['LSTM', 'LSTM', 'LSTM', 'LSTM'],
    'rmse': [33.42, 6.15, 7.66, 9.03],
    'mae': [21.53, 4.74, 5.57, 6.87],
    'r2_score': [0.8050, 0.7416, 0.8773, 0.8087],
    'confidence': [54.6, 73.8, 80.3, 69.4]
}

metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_sql('model_metrics', engine, if_exists='append', index=False)
print("model_metrics table imported:")
print(metrics_df)

model_metrics table imported:
        city model_type   rmse    mae  r2_score  confidence
0      Delhi       LSTM  33.42  21.53    0.8050        54.6
1  Bengaluru       LSTM   6.15   4.74    0.7416        73.8
2     Mumbai       LSTM   7.66   5.57    0.8773        80.3
3  Hyderabad       LSTM   9.03   6.87    0.8087        69.4


In [9]:
# Import Anomalies 
import pickle

with open('../src/models/city_importances.pkl', 'rb') as f:
    city_importances = pickle.load(f)

with open('../src/models/confidence_results.pkl', 'rb') as f:
    confidence_results = pickle.load(f)

source_rows = []

for city, importance in city_importances.items():
    top3 = importance.sort_values(ascending=False).head(3)
    source_rows.append({
        'city': city,
        'top_source_1': top3.index[0],
        'top_source_2': top3.index[1],
        'top_source_3': top3.index[2],
        'pm10_importance': float(importance.get('PM10', 0)),
        'co_importance': float(importance.get('CO', 0)),
        'nh3_importance': float(importance.get('NH3', 0)),
        'confidence': float(confidence_results[city])
    })

source_df = pd.DataFrame(source_rows)
source_df.to_sql('source_analysis', engine, if_exists='append', index=False)
print("source_analysis table imported:")
print(source_df)

source_analysis table imported:
        city top_source_1 top_source_2 top_source_3  pm10_importance  \
0      Delhi           CO         PM10   season_enc         0.230322   
1  Bengaluru         PM10           NO          SO2         0.447822   
2     Mumbai         PM10          SO2      Benzene         0.541160   
3  Hyderabad         PM10   season_enc           CO         0.337429   

   co_importance  nh3_importance  confidence  
0       0.488130        0.032297    0.546298  
1       0.059502        0.038369    0.737605  
2       0.017202        0.039070    0.802883  
3       0.093046        0.051459    0.693812  


In [11]:
# Verify All Tables
import pickle

with open('../src/models/top_anomalies.pkl', 'rb') as f:
    top_anomalies = pickle.load(f)

all_anomalies = []

for city, anomaly_df in top_anomalies.items():
    temp = anomaly_df.copy()
    temp['city'] = city
    temp['is_anomaly'] = True
    temp['severity'] = temp['PM25'].apply(
        lambda x: 'Severe' if x > 200 else ('High' if x > 100 else 'Moderate')
    )
    temp['reconstruction_error'] = temp['error']
    temp['threshold'] = 0.0
    temp = temp[['city', 'date', 'PM25', 'reconstruction_error', 'threshold', 'is_anomaly', 'severity']]
    
    # Rename to lowercase
    temp.columns = [c.lower() for c in temp.columns]
    all_anomalies.append(temp)

anomalies_df = pd.concat(all_anomalies, ignore_index=True)
anomalies_df.to_sql('anomalies', engine, if_exists='append', index=False)
print("anomalies table imported:", len(anomalies_df), "rows")
print(anomalies_df.head())

anomalies table imported: 40 rows
    city       date        pm25  reconstruction_error  threshold  is_anomaly  \
0  Delhi 2017-11-08  647.718036              0.048819        0.0        True   
1  Delhi 2019-11-03  551.715569              0.047747        0.0        True   
2  Delhi 2020-11-09  502.356355              0.037887        0.0        True   
3  Delhi 2017-11-07  512.595736              0.027492        0.0        True   
4  Delhi 2021-11-05  413.793297              0.025753        0.0        True   

  severity  
0   Severe  
1   Severe  
2   Severe  
3   Severe  
4   Severe  
